---
title: "Tool Calling and Execution Loops"
description: "Train a model to emit schema-valid tool calls, decide when not to call, and recover from malformed results."
categories: [machine-learning, posttraining, tools]
---

Tool calling is a posttraining outcome, not a prompt-engineering feature bolted onto a finished model. The model must choose among available tools, emit arguments that satisfy a typed schema, decide when answering directly is better than calling, read results back, and continue within a step budget. Each of those is a separate learned behavior with a separate failure mode, and the evaluation protocol keeps them separate.

## Typed traces and the execution loop

Extend the Chapter 08 chat template with tool roles: an assistant message carries a tool name and JSON arguments, and a tool message carries the execution result. The tool inventory reaches the prompt as typed schemas. An episode interleaves assistant turns, tool calls, and tool results:

```json
{
  "role": "assistant",
  "tool_calls": [
    {"name": "get_weather", "arguments": {"city": "Oslo", "unit": "celsius"}}
  ]
}
```

The loop is: generate, parse, validate against the schema, execute against a registry of deterministic fake tools (weather, calculator, file lookup, calendar), append the result, repeat until a final answer or the step budget is exhausted. Malformed calls are training data, not exceptions: return a structured error message and record whether the next turn recovers. Training loss applies to tool-call spans only, using the masked cross-entropy from Chapter 08; compare free decoding with post-hoc validation against JSON-schema-constrained decoding for the argument span.

## What a correct call requires

A tool call is correct only if three independent conditions hold, and each has its own metric:

| Condition | Failure mode | Metric |
|---|---|---|
| The tool exists and fits the request | hallucinated or wrong tool | exact tool-choice accuracy |
| Arguments parse and satisfy the schema | schema violation | schema-validity rate |
| Calling was the right decision | unnecessary call | when-to-call accuracy, false-call rate |

A model can score perfectly on one row and fail the others, which is why a single "tool accuracy" number is uninterpretable.

## Evaluation protocol

On held-out requests, report exact tool choice and argument correctness, schema-validity rate, when-to-call accuracy on prompts where calling helps and prompts where it does not, hallucinated-tool and hallucinated-parameter rates, recovery success after a deliberately corrupted tool result, multi-turn completion within the step budget, and the calibration curve of call decisions. Hold out tools and their schemas entirely from training, not just their names: generalization to an unseen schema is a different claim from generalization to an unseen phrasing of a seen tool.

## Reliability connection

Each failure mode in this chapter maps to one from the course vocabulary: a hallucinated API is proxy optimization (the model learned the *shape* of calls from traces), a schema violation is a specification error, an unnecessary call is a calibration failure, and ignoring a corrected tool result is a correction failure. The protocol above measures each separately, and it carries into the Chapter 14 capstone unchanged.

## Summary

Tool training is SFT over scripted trajectories with call-span masks, optionally sharpened by DPO pairs on borderline prompts (correct call versus hallucinated tool, well-scoped call versus unnecessary call). The evaluation protocol, not the training loss, defines what was actually learned.


### [P11.1] Correct tool-call conditions

List the three independent conditions for a correct tool call and one metric for each.

In [ ]:
#| echo: false
#| eval: false
#| output: false
# Gur gbby zhfg or nccebcevngr naq rkvfg, zrnfherq ol rknpg gbby-pubvpr npphenpl; nethzragf zhfg fngvfsl gur fpurzn, zrnfherq ol fpurzn-inyvqvgl engr; naq pnyyvat zhfg or gur evtug qrpvfvba, zrnfherq ol jura-gb-pnyy npphenpl be snyfr-pnyy engr.

### [P11.2] Separating tool-call failures

Design three held-out episodes that separate wrong-tool selection from correct-tool malformed arguments from unnecessary calls. For each, state which metric from the protocol fails and which still passes.

In [ ]:
#| echo: false
#| eval: false
#| output: false
# Rcvfbqr bar: n erdhrfg gung arrqf nevguzrgvp jurer gur zbqry pnyyf trg_jrngure. Gbby pubvpr vf jebat, fb rknpg gbby-pubvpr npphenpl snvyf, ohg vs gur nethzragf unccra gb fngvfsl gur jrngure fpurzn, fpurzn-inyvqvgl fgvyy cnffrf. Rcvfbqr gjb: n jrngure erdhrfg jurer gur zbqry pnyyf trg_jrngure jvgu n zvffvat erdhverq pvgl svryq. Gbby pubvpr vf pbeerpg, fb gbby-pubvpr npphenpl cnffrf, juvyr gur fpurzn-inyvqvgl engr snvyf. Rcvfbqr guerr: n dhrfgvba nafjrenoyr sebz gur zbqry bja xabjyrqtr, fhpu nf gur pncvgny bs Abejnl, jurer gur zbqry fgvyy pnyyf n gbby. Fpurzn naq pubvpr znl obgu or svar, fb bayl jura-gb-pnyy npphenpl naq snyfr-pnyy engr pngpu vg.